> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 4 — Claim-Evidence & Abstention (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2004.%20Hallucinations%20and%20RAG%20Systems/LC4LSH_Chapter_4_Claim_Evidence_and_Abstention.ipynb)

**Learning objectives**
- Decompose an answer into atomic claims
- Link each claim to supporting evidence spans
- Validate citations and surface contradictions
- Abstain (INSUFFICIENT_EVIDENCE) when support is missing

> Runtime: ~3 min (CPU)  
> Cost: $0 (optional LLM judge gated)  
> Data: synthetic evidence + claims


A RAG answer is trustworthy only if **every claim is backed by a retrievable evidence span**.

Pipeline: split answer into atomic claims -> retrieve best evidence per claim -> validate each citation -> **abstain** when support is missing. This is the core anti-hallucination pattern for scientific and clinical QA.


## API keys & credentials

Mostly local (no paid API needed); the bootstrap sets up an optional provider for gated LLM cells.


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print("API keys loaded for", API_KEY_PROVIDER)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


## Installation (pinned)


In [ ]:
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "pydantic>=2.6,<3" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter4-claim-evidence"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")


## An evidence store (typed)


In [ ]:
from pydantic import BaseModel, Field
class EvidenceSpan(BaseModel):
    span_id: str
    doc_id: str
    text: str
    section: str = ""
EVIDENCE = [
    EvidenceSpan(span_id="E1", doc_id="D1", text="Metformin reduces hepatic glucose production.", section="Mechanism"),
    EvidenceSpan(span_id="E2", doc_id="D2", text="In trial NCT04280705, metformin reduced diabetes incidence by 31% vs placebo.", section="Results"),
    EvidenceSpan(span_id="E3", doc_id="D3", text="Aspirin irreversibly acetylates COX-1 at serine 530.", section="Mechanism"),
    EvidenceSpan(span_id="E4", doc_id="D5", text="GLP-1 agonists reduced HbA1c by 1.0-1.5% in phase 3 trials.", section="Results"),
]
print(len(EVIDENCE), "evidence spans")


## 1. Atomic claims with citations


In [ ]:
class Claim(BaseModel):
    claim_id: str
    text: str
    cited_span_ids: list[str] = Field(default_factory=list)
ANSWER_CLAIMS = [
    Claim(claim_id="C1", text="Metformin lowers hepatic glucose output.", cited_span_ids=["E1"]),
    Claim(claim_id="C2", text="Metformin cut diabetes incidence by 31% in NCT04280705.", cited_span_ids=["E2"]),
    Claim(claim_id="C3", text="Aspirin reversibly blocks COX-1.", cited_span_ids=["E3"]),
    Claim(claim_id="C4", text="Metformin cures type 1 diabetes.", cited_span_ids=[]),
]
print(len(ANSWER_CLAIMS), "claims extracted")


## 2. Validate each citation (deterministic harness)


In [ ]:
SPAN = {e.span_id: e for e in EVIDENCE}
NEG = {"irreversibly": "reversibly", "reduces": "abolishes", "inhibits": "activates"}
def validate(claim):
    if not claim.cited_span_ids:
        return {"id": claim.claim_id, "verdict": "UNSUPPORTED", "reason": "no citation"}
    for sid in claim.cited_span_ids:
        span = SPAN.get(sid)
        if span is None:
            return {"id": claim.claim_id, "verdict": "INVALID_CITATION", "reason": sid + " not in store"}
        for a, b in NEG.items():
            if a in span.text.lower() and b in claim.text.lower():
                return {"id": claim.claim_id, "verdict": "CONTRADICTED", "reason": f"span says '{a}', claim says '{b}'"}
    return {"id": claim.claim_id, "verdict": "SUPPORTED", "reason": "consistent"}
for c in ANSWER_CLAIMS:
    print(validate(c))


## 3. Abstention policy


In [ ]:
def answer_with_abstention(claims):
    verdicts = [validate(c)["verdict"] for c in claims]
    if any(v in ("UNSUPPORTED", "CONTRADICTED", "INVALID_CITATION") for v in verdicts):
        bad = [c.text for c, v in zip(claims, verdicts) if v != "SUPPORTED"]
        return "INSUFFICIENT_EVIDENCE\nProblematic claims:\n- " + "\n- ".join(bad)
    return "\n".join("- " + c.text + " [" + ",".join(c.cited_span_ids) + "]" for c in claims)
print(answer_with_abstention(ANSWER_CLAIMS))
print("--- clean ---")
print(answer_with_abstention([c for c in ANSWER_CLAIMS if validate(c)["verdict"] == "SUPPORTED"]))


## 4. (Optional) LLM-as-judge - gated


In [ ]:
RUN_LLM_JUDGE = False
if RUN_LLM_JUDGE:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = ChatPromptTemplate.from_template("Does EVIDENCE entail CLAIM? SUPPORTS/CONTRADICTS/NEUTRAL.\nEVIDENCE: {e}\nCLAIM: {c}")
    c = ANSWER_CLAIMS[2]
    print(judge.invoke(prompt.format_messages(e=SPAN[c.cited_span_ids[0]].text, c=c.text)).content)
else:
    print("LLM judge skipped (RUN_LLM_JUDGE=False).")


## Limitations & safety notes

- **Keyword proxy** misses subtle contradictions; use a trained NLI model or LLM judge in production.
- **Abstention is a feature.** 'I don't know' is safer than fabrication.
- **Citation != correctness.** Verify the span supports the specific claim.
- **Clinical stakes.** Never let an unvalidated claim reach a patient-facing output.


In [ ]:
# Cleanup
import gc
for _v in ("EVIDENCE", "ANSWER_CLAIMS", "SPAN"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why decompose into atomic claims before validating?</summary>Long answers mix supported and unsupported statements; per-claim validation isolates which lack evidence.</details>

<details><summary>UNSUPPORTED vs CONTRADICTED?</summary>UNSUPPORTED = no adequate evidence; CONTRADICTED = evidence disagrees. Both trigger abstention.</details>

<details><summary>Why is keyword entailment insufficient clinically?</summary>It misses paraphrase, hedging, dosage and numeric errors.</details>

### Tasks
- **Task A** - Replace the keyword proxy with `cross-encoder/nli-deberta-v3-base`.
- **Task B** - Add a numeric-consistency check flagging number mismatches.
- **Task C** - Build a contradiction surfacer returning the conflicting span text.
- **Task D** - Add a repair step rewriting a contradicted claim to match its evidence.
